In [1]:
import pandas as pd
import numpy as np
import pickle
import category_encoders as ce
import math
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
import copy
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt

In [3]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
#df = df.reset_index(drop=True)
df = pd.read_pickle("Data/df_preprocessed_13_11.pickle")

In [6]:
def filter_outliers(df, upper, lower, target_col = "award_contract/val_total"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers, with high amount = {}, and low amount = {}".format(len(outlier_indices), high_amount, low_amount))
    df = df.drop(labels = outlier_indices, axis = 0)
    return df

In [7]:
def train_validate_test_split(df, train, validate):
    target_col = "award_contract/val_total"
    df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df))
    test_indice = int((1-validate-train) * len(df))

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice: test_indice]

    X_train = train_set.drop(columns = [target_col]).values
    y_train = train_set[target_col].values

    X_val = val_set.drop(columns = [target_col]).values
    y_val = val_set[target_col].values

    X_test = test_set.drop(columns = [target_col]).values
    y_test = test_set[target_col].values

    return X_train, y_train, X_val, y_val, X_test, y_test

In [8]:
def encode_data(df, encoding, target_column="award_contract/val_total"):
    
    base_n_encoder_cols = ["cpv_code", "country", "language"]
    one_hot_cols = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "ca_type", 
                "ca_activity", "procedure"]
    numerical_cols = ["duration", "nb_tenders_received", "nb_tenders_received_sme", "ac_price", "ac_weighting", "ac_cost/ac_weighting"]
    text_cols = ["short_descr", "ac_criterion", "object_contract/title", "object_descr/title", "ac_cost/ac_criterion"]

    if encoding == "onehot":
        df_encoded = pd.get_dummies(df, columns=df.drop(columns = ["award_contract/val_total", "date_conclusion_contract"]).columns, drop_first=True, dtype=int)
        df_encoded = df_encoded.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
        X_train, X_test = np.split(df_encoded.drop(columns=["award_contract/val_total", "date_conclusion_contract"]).values, [int(0.8 * len(df))])
        y_train, y_test = np.split(df_encoded["award_contract/val_total"].values, [int(0.8 * len(df))])
    
    elif encoding == "onehot and baseN":
        encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)
        df = encoder.fit_transform(df)
        df = pd.get_dummies(df, columns=(one_hot_cols), drop_first=True, dtype=int)

    return df

In [9]:
def retrieve_selection(df, selected_columns):
    # Select rows where all specified columns are not empty
    df_selection = df[df[selected_columns].notna().all(axis=1)].reset_index(drop=True)
    
    return df_selection

In [10]:
def impute_scale(df, with_encoding = False):
    base_n_encoder_cols = ["cpv_code", "country", "language"]
    one_hot_cols = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "ca_type", 
                "ca_activity"]
    numerical_cols = ["duration", "nb_tenders_received", "nb_tenders_received_sme", "ac_price", "ac_weighting", "ac_cost/ac_weighting"]
    cat_cols = base_n_encoder_cols + one_hot_cols

    cat_imputer = SimpleImputer(strategy="most_frequent", missing_values=np.nan)
    #num_imputer = SimpleImputer(strategy = "mean", missing_values=np.nan)
    num_imputer = KNNImputer(n_neighbors=10, missing_values=np.nan)
    scaler = RobustScaler()

    if with_encoding == True:
        for col in cat_cols:
            if pd.isna(df[col]).any() == True:
                data = df[col].values.reshape(-1,1)
                imputed_data = cat_imputer.fit_transform(data)
                df[col] = imputed_data.flatten()
    else:
        print("no encoding was applied because encoding is set to false")
        
    numerical_cols = df[numerical_cols].columns[df[numerical_cols].isnull().sum() < len(df)].to_list()
    numeric_data = df[numerical_cols].values.reshape((len(df), len(numerical_cols)))
    imputed_data_num = num_imputer.fit_transform(numeric_data)
    scaled_and_imputed_num_data = scaler.fit_transform(imputed_data_num)
    df_numeric = pd.DataFrame(data=scaled_and_imputed_num_data,columns = numerical_cols)
    df[numerical_cols] = df_numeric

    return df, numerical_cols, numeric_data, imputed_data_num, scaled_and_imputed_num_data

In [11]:
def filter_rows(df, condition_dict):
    for feature_type in condition_dict.keys():
        for feature in condition_dict[feature_type]:
            if feature_type == "categorical":
                unique_values = df[feature].value_counts().keys()

                outlier_values = []
                for value in unique_values:
                    if len(df.loc[df[feature] == value]) / len(df) <= condition_dict[feature_type][feature]:
                        outlier_values.append(value)

                removal_list = []
                for i in range(0, len(df)):
                    if df[feature].iloc[i] in outlier_values:
                        removal_list.append(i)
                df = df.drop(labels = removal_list, axis = 0).reset_index(drop=True)

            elif feature_type == "numerical":
                df = filter_outliers(df, target_col=feature, lower=condition_dict[feature_type][feature][0], 
                                     upper = condition_dict[feature_type][feature][1])
    return df

In [ ]:
#define selections
#selection_1 =  ["duration"]#["cpv_code", "country", "type_contract", "procedure", "ca_type", "ca_activity", "nb_tenders_received", "duration"]
#retrieve selection
#df = retrieve_selection(df, selection_1)

df, numerical_cols, numeric_data, imputed_data_num, scaled_and_imputed_num_data = impute_scale(df)

#filter based on selection criteria 
condition_dict = {"categorical":{"language": 0.05,
                                 "cpv_code": 0.05,
                                 },
                 "numeric": {"award_contract/val_total": [0.15, 0.95]
                             }}#df = filter_rows(df, condition_dict=condition_dict)
df= encode_data(df, encoding="onehot and baseN")

In [ ]:
len(df

In [ ]:
pd.isna(df).any()

In [ ]:
numerical_cols = ["duration", "nb_tenders_received", "nb_tenders_received_sme", "ac_price", "ac_weighting", "ac_cost/ac_weighting"]


In [ ]:
df.head()

In [ ]:
len(df.loc[df["duration"] != 0])

In [29]:
scaler = RobustScaler()
scaled_and_imputed_num_data = scaler.fit_transform(df.values)
df_numeric = pd.DataFrame(data=scaled_and_imputed_num_data,columns = numerical_cols)

In [ ]:
df_numeric.head()

In [12]:
with open("Data/train_val_test/set_1", "wb") as f:
    pickle.dump(df,f)